# Stage 03: mTLS Auth

This stage uses the agent to generate code that reads protected tenant-scoped text from the `internal-db` service. The service is simple on purpose: the point is that the same code stays blocked until the pod identity is allowed through an `AuthorizationPolicy`.


In [ ]:
from __future__ import annotations

import contextlib
import io
import os
import subprocess
import textwrap
from typing import Any

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from utils import extract_output

import nest_asyncio

nest_asyncio.apply()


## Ask the agent to write a natural internal-context reader

There are no credentials in the code. The goal is just to fetch a protected internal text response and present it to the user.


In [ ]:
LITELLM_BASE_URL = 'http://llm-gateway.workshop-system.svc.cluster.local:4000/v1'
LITELLM_MODEL = 'workshop-gemini'
LITELLM_API_KEY = 'not-needed'
INTERNAL_DB_URL = 'http://internal-db:8080/secure-data'
POLICY_NAME = 'allow-vscode-to-internal-db'

with open('/var/run/secrets/kubernetes.io/serviceaccount/namespace', 'r') as f:
    WORKSPACE_NAMESPACE = f.read().strip()

INTERNAL_CONTEXT_PROMPT = textwrap.dedent(
    f"""
    Write Python code using only the standard library that fetches plain text from {INTERNAL_DB_URL}.

    Store the response body in a variable named internal_context.
    Print a friendly sentence for the user that includes the internal context.

    Return only Python code. Do not add Markdown fences.
    """
).strip()

code_generation_agent = Agent(
    OpenAIChatModel(
        LITELLM_MODEL,
        provider=OpenAIProvider(
            base_url=LITELLM_BASE_URL,
            api_key=LITELLM_API_KEY,
        ),
    ),
    system_prompt='You are a helpful Python automation assistant. Return only Python code.',
)

generated_code = extract_output(code_generation_agent.run_sync(INTERNAL_CONTEXT_PROMPT))
print(generated_code)


## Run the generated code before the allow policy


In [ ]:
def run_generated_code(code: str) -> dict[str, Any]:
    exec_env: dict[str, Any] = {}
    stdout = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, exec_env, exec_env)
    except BaseException as exc:
        return {
            'stdout': stdout.getvalue(),
            'error': f'{type(exc).__name__}: {exc}',
            'internal_context': exec_env.get('internal_context'),
        }

    return {
        'stdout': stdout.getvalue(),
        'error': None,
        'internal_context': exec_env.get('internal_context'),
    }


In [ ]:
run_generated_code(generated_code)


## Allow this pod identity to reach the internal service


In [ ]:
service_account = subprocess.check_output(
    [
        'kubectl',
        'get',
        'pod',
        os.environ['HOSTNAME'],
        '-n',
        WORKSPACE_NAMESPACE,
        '-o',
        'jsonpath={.spec.serviceAccountName}',
    ],
    text=True,
).strip()

principal = f'cluster.local/ns/{WORKSPACE_NAMESPACE}/sa/{service_account}'

ALLOW_INTERNAL_DB_POLICY = textwrap.dedent(
    f"""
    apiVersion: security.istio.io/v1beta1
    kind: AuthorizationPolicy
    metadata:
      name: {POLICY_NAME}
    spec:
      selector:
        matchLabels:
          app: internal-db
      action: ALLOW
      rules:
      - from:
        - source:
            principals:
            - {principal}
    """
).strip()

print(ALLOW_INTERNAL_DB_POLICY)


In [ ]:
apply_result = subprocess.run(
    ['kubectl', 'apply', '-f', '-'],
    input=ALLOW_INTERNAL_DB_POLICY,
    text=True,
    capture_output=True,
    check=True,
)
print(apply_result.stdout or f'{POLICY_NAME} applied')


In [ ]:
print(
    subprocess.check_output(
        [
            'kubectl',
            'get',
            'authorizationpolicies.security.istio.io',
            POLICY_NAME,
            '-n',
            WORKSPACE_NAMESPACE,
            '-o',
            'yaml',
        ],
        text=True,
    )
)


In [ ]:
run_generated_code(generated_code)


## Cleanup the allow policy


In [ ]:
delete_result = subprocess.run(
    [
        'kubectl',
        'delete',
        'authorizationpolicy',
        POLICY_NAME,
        '-n',
        WORKSPACE_NAMESPACE,
        '--ignore-not-found=true',
    ],
    text=True,
    capture_output=True,
    check=True,
)
print(delete_result.stdout or f'{POLICY_NAME} deleted')
